# Параллельное ретро-сравнение прогнозов кассовых показателей

Самодостаточный ноутбук для ретро-прогона за `RETRO_DATE_FROM..RETRO_DATE_TO`. Для каждой даты скоринга используются только данные до `T−2`, строится 31 модельный шаг и сохраняются 30 дат `score_date..score_date+29`.

Кассы одной даты считаются параллельно. SARIMA проходит единый набор проверок устойчивости; отклонённые модели заменяются fallback по дням недели. Для NS за один bootstrap рассчитывается сетка квантилей 0.01–0.15, после чего квантиль автоматически калибруется под долю невыдач старого алгоритма.

In [ ]:
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from joblib import Parallel, delayed, parallel_backend
from prefect.blocks.system import Secret
from sqlalchemy import text
from statsmodels.tsa.statespace.sarimax import SARIMAX
from toolbox import oracle

warnings.filterwarnings("ignore")

## 1. Параметры

Период ретро универсален: границы, окно истории, лаг и горизонт задаются ниже. `SELECTED_QUANTILE=None` включает автоматический выбор; значение из сетки позволяет переиспользовать уже рассчитанные прогнозы без повторного обучения моделей.

In [ ]:
SOURCE_TABLE = "AIDA2.AIDA_TRS_DTM_CASHOP@aida"
OLD_FORECAST_TABLE = "AIDA2.AIDA_TRS_DTM_FORECAST_HISTORY@aida"
ORACLE_TARGET_TABLE = "EMA_CASHDESK_PREDS_2M"

RETRO_DATE_FROM = pd.Timestamp("2026-04-15")
RETRO_DATE_TO = pd.Timestamp("2026-06-15")
HISTORY_MONTHS = 12
ACTIVE_LOOKBACK_MONTHS = 1
DATA_LAG_DAYS = 2
FORECAST_DAYS = 30
MODEL_STEPS = FORECAST_DAYS + DATA_LAG_DAYS - 1

INITIAL_TRAIN_DAYS = 60
BACKTEST_STEP_DAYS = 7
BOOTSTRAP_ITERATIONS = 1000
BOOTSTRAP_QUANTILES = np.round(np.arange(0.01, 0.16, 0.01), 2)
MIN_BACKTEST_ERROR_BLOCKS = 5
MIN_SARIMA_DAYS = 90
SARIMA_NO_ERROR_HISTORY_ADJUSTMENT = 0.15
SARIMA_ORDER = (1, 1, 1)
SARIMA_SEASONAL_ORDER = (1, 0, 1, 7)
CASH_NEED_CLIP_UPPER = 0.0
FORECAST_SCALE_MULTIPLIER = 20.0

EXCLUDE_DATE_RANGES = [("2025-09-01", "2025-11-01")]
CASHDESK_FILTER = None
RANDOM_SEED = 42
N_JOBS = 16
SELECTED_QUANTILE = None

score_dates = pd.date_range(RETRO_DATE_FROM, RETRO_DATE_TO, freq="D")
MIN_REPORT_DATE = score_dates.min() - pd.Timedelta(days=DATA_LAG_DAYS)
DATA_DATE_FROM = MIN_REPORT_DATE - pd.DateOffset(months=HISTORY_MONTHS)
FACT_DATE_TO_EXCLUSIVE = score_dates.max() + pd.Timedelta(days=FORECAST_DAYS)
OLD_DATE_TO_EXCLUSIVE = RETRO_DATE_TO + pd.Timedelta(days=1)

LOAD_RAW_FROM_CACHE = False
RAW_CACHE_PATH = Path("data/raw/cashdesk_retro_2m_raw.parquet")
OLD_CACHE_PATH = Path("data/raw/cashdesk_old_forecast_2m.parquet")
OUTPUT_DIR = Path("forecast_results/retro_parallel_nan_guarded_qgrid_v1")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
USE_CHECKPOINTS = True

QUANTILE_COLUMNS = {
    float(quantile): "atdtmco_ns_pred_q{:02d}".format(int(round(100 * quantile)))
    for quantile in BOOTSTRAP_QUANTILES
}

print("Дат скоринга: {}".format(len(score_dates)))
print("История и факты: {} — {}".format(
    DATA_DATE_FROM.date(),
    (FACT_DATE_TO_EXCLUSIVE - pd.Timedelta(days=1)).date(),
))

## 2. Загрузка и подготовка данных

Oracle и parquet-кэши читаются только в основном процессе. Английский ключ исходной таблицы `atdtmco_cashdesk_name_trn` сопоставляется с `cashdesk_name` старой истории. `forecast_model` переводится на общую отрицательную шкалу. Контроль `flow_minimum` сохранён в одном показателе совпадения.

In [ ]:
USERNAME_CDW = "sb_analytics"
engine_cdw = None


async def create_cdw_engine():
    password_cdw = (await Secret.load("pass-sb-analytics")).get()
    return oracle.create_engine_cdw(USERNAME_CDW, password_cdw)


if LOAD_RAW_FROM_CACHE:
    raw_df = pd.read_parquet(RAW_CACHE_PATH)
    old_history_df = pd.read_parquet(OLD_CACHE_PATH)
else:
    engine_cdw = await create_cdw_engine()
    source_query = """
    select
        atdtmco_cashdesk_name,
        atdtmco_cashdesk_name_trn,
        atdtmco_calday,
        atdtmco_saldo_turn,
        atdtmco_ns
    from {source_table}
    where atdtmco_calday >= date '{date_from}'
      and atdtmco_calday < date '{date_to}'
    """.format(
        source_table=SOURCE_TABLE,
        date_from=DATA_DATE_FROM.date().isoformat(),
        date_to=FACT_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        source_query += (
            "\n  and not (atdtmco_calday >= date '{}' and atdtmco_calday < date '{}')"
            .format(exclude_start, exclude_end)
        )
    if CASHDESK_FILTER:
        escaped_names = ", ".join(
            "'" + name.replace("'", "''") + "'" for name in CASHDESK_FILTER
        )
        source_query += "\n  and atdtmco_cashdesk_name in ({})".format(escaped_names)

    old_query = """
    select cashdesk_name, calday, flow_minimum, forecast_model, forecast_time
    from {old_table}
    where calday >= date '{date_from}'
      and calday < date '{date_to}'
    """.format(
        old_table=OLD_FORECAST_TABLE,
        date_from=RETRO_DATE_FROM.date().isoformat(),
        date_to=OLD_DATE_TO_EXCLUSIVE.date().isoformat(),
    )
    with engine_cdw.connect() as conn:
        raw_df = pd.read_sql(text(source_query), conn)
        old_history_df = pd.read_sql(text(old_query), conn)

    RAW_CACHE_PATH.parent.mkdir(parents=True, exist_ok=True)
    raw_df.to_parquet(RAW_CACHE_PATH, index=False)
    old_history_df.to_parquet(OLD_CACHE_PATH, index=False)

raw_df.columns = raw_df.columns.str.lower()
old_history_df.columns = old_history_df.columns.str.lower()
raw_df["atdtmco_calday"] = pd.to_datetime(raw_df["atdtmco_calday"]).dt.normalize()
old_history_df["calday"] = pd.to_datetime(old_history_df["calday"]).dt.normalize()
old_history_df["forecast_time"] = pd.to_datetime(
    old_history_df["forecast_time"], errors="coerce"
)

daily_df = (
    raw_df
    .sort_values(["atdtmco_cashdesk_name", "atdtmco_calday"])
    .groupby(
        ["atdtmco_cashdesk_name", "atdtmco_calday"],
        as_index=False,
        dropna=False,
    )
    .agg(
        atdtmco_cashdesk_name_trn=("atdtmco_cashdesk_name_trn", "last"),
        atdtmco_saldo_turn_fact=("atdtmco_saldo_turn", "sum"),
        atdtmco_ns_daily_min_raw=("atdtmco_ns", "min"),
    )
    .rename(columns={"atdtmco_calday": "calday"})
    .sort_values(["atdtmco_cashdesk_name", "calday"])
    .reset_index(drop=True)
)
daily_df["atdtmco_ns_fact"] = daily_df["atdtmco_ns_daily_min_raw"].clip(upper=0.0)

old_history_dedup_df = (
    old_history_df
    .sort_values("forecast_time")
    .drop_duplicates(["cashdesk_name", "calday"], keep="last")
    .reset_index(drop=True)
)
old_history_dedup_df["atdtmco_ns_pred_old"] = -pd.to_numeric(
    old_history_dedup_df["forecast_model"], errors="coerce"
)

flow_check_df = old_history_dedup_df[
    ["cashdesk_name", "calday", "flow_minimum"]
].merge(
    daily_df[
        ["atdtmco_cashdesk_name_trn", "calday", "atdtmco_ns_daily_min_raw"]
    ],
    left_on=["cashdesk_name", "calday"],
    right_on=["atdtmco_cashdesk_name_trn", "calday"],
    how="inner",
)
flow_minimum_match_rate = np.isclose(
    pd.to_numeric(flow_check_df["flow_minimum"], errors="coerce"),
    flow_check_df["atdtmco_ns_daily_min_raw"],
    atol=0.01,
    rtol=0.0,
    equal_nan=False,
).mean()
print("Совпадение дневного min(NS) с flow_minimum: {:.1%}".format(
    flow_minimum_match_rate
))

## 3. Модельное ядро: только nan-guarded SARIMA

Исключённый интервал `2025-09-01 <= date < 2025-11-01` остаётся `NaN`; обычные пропущенные дни кассы считаются нулевыми. И финальная модель, и каждый rolling backtest fit отклоняются при несходимости, неустойчивых AR/MA-корнях, нечисловом либо несоразмерном прогнозе. При отклонении финальной SARIMA используется fallback по дням недели.

In [ ]:
@dataclass(frozen=True)
class SarimaConfig:
    order: Tuple[int, int, int]
    seasonal_order: Tuple[int, int, int, int]
    maxiter: int = 200


SARIMA_CONFIG = SarimaConfig(
    order=SARIMA_ORDER,
    seasonal_order=SARIMA_SEASONAL_ORDER,
)


def make_regular_daily_series(
    cashdesk_df: pd.DataFrame,
    value_col: str,
    report_date: pd.Timestamp,
) -> pd.Series:
    observed = (
        cashdesk_df.set_index("calday")[value_col].sort_index().astype(float)
    )
    full_index = pd.date_range(observed.index.min(), report_date, freq="D")
    series = observed.reindex(full_index).fillna(0.0)
    for exclude_start, exclude_end in EXCLUDE_DATE_RANGES:
        excluded_mask = (
            (series.index >= pd.Timestamp(exclude_start))
            & (series.index < pd.Timestamp(exclude_end))
        )
        series.loc[excluded_mask] = np.nan
    series.index.name = "calday"
    return series


def make_future_index(y: pd.Series, steps: int) -> pd.DatetimeIndex:
    return pd.date_range(
        y.index.max() + pd.Timedelta(days=1), periods=steps, freq="D"
    )


def guarded_sarima_forecast(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> Tuple[object, np.ndarray]:
    model = SARIMAX(
        y,
        order=config.order,
        seasonal_order=config.seasonal_order,
        enforce_stationarity=True,
        enforce_invertibility=True,
    )
    fitted = model.fit(disp=False, maxiter=config.maxiter)
    if not bool(fitted.mle_retvals.get("converged", False)):
        raise ValueError("SARIMA не сошлась")

    for root_name, roots in (("AR", fitted.arroots), ("MA", fitted.maroots)):
        root_modulus = np.abs(np.asarray(roots, dtype=complex))
        if root_modulus.size and (
            not np.isfinite(root_modulus).all() or (root_modulus <= 1.0).any()
        ):
            raise ValueError("Неустойчивые {}-корни".format(root_name))

    forecast = fitted.get_forecast(steps=steps).predicted_mean.to_numpy(dtype=float)
    valid_history = y.dropna().to_numpy(dtype=float)
    history_scale = max(
        1.0,
        float(np.quantile(np.abs(valid_history), 0.99)),
    )
    if not np.isfinite(forecast).all():
        raise ValueError("Нечисловой SARIMA-прогноз")
    if (np.abs(forecast) > FORECAST_SCALE_MULTIPLIER * history_scale).any():
        raise ValueError("Несоразмерный SARIMA-прогноз")
    return fitted, forecast


def weekday_point_forecast(y: pd.Series, steps: int) -> np.ndarray:
    y_clean = y.dropna()
    global_value = float(y_clean.median())
    weekday_values = y_clean.groupby(y_clean.index.dayofweek).median()
    return np.asarray(
        [
            weekday_values.get(day.dayofweek, global_value)
            for day in make_future_index(y, steps)
        ],
        dtype=float,
    )


def weekday_quantile_values(
    y: pd.Series,
    steps: int,
    quantile: float,
) -> np.ndarray:
    y_clean = y.dropna()
    global_quantile = float(y_clean.quantile(quantile))
    weekday_quantiles = y_clean.groupby(y_clean.index.dayofweek).quantile(quantile)
    values = np.asarray(
        [
            weekday_quantiles.get(day.dayofweek, global_quantile)
            for day in make_future_index(y, steps)
        ],
        dtype=float,
    )
    return np.minimum(values, CASH_NEED_CLIP_UPPER)


def collect_backtest_error_blocks(
    y: pd.Series,
    steps: int,
    config: SarimaConfig,
) -> np.ndarray:
    error_blocks = []
    max_train_end = len(y) - steps
    for train_end in range(
        INITIAL_TRAIN_DAYS,
        max_train_end + 1,
        BACKTEST_STEP_DAYS,
    ):
        train = y.iloc[:train_end]
        test = y.iloc[train_end:train_end + steps]
        if train.notna().sum() < INITIAL_TRAIN_DAYS:
            continue
        try:
            _, forecast = guarded_sarima_forecast(train, steps, config)
        except Exception:
            continue
        error = test.to_numpy(dtype=float) - forecast
        if np.isfinite(error).all():
            error_blocks.append(error)
    if not error_blocks:
        return np.empty((0, steps), dtype=float)
    return np.vstack(error_blocks)


def forecast_ns_quantile_grid(
    y: pd.Series,
    steps: int,
    random_generator: np.random.Generator,
) -> Dict[str, np.ndarray]:
    error_blocks = collect_backtest_error_blocks(y, steps, SARIMA_CONFIG)
    _, mean_values = guarded_sarima_forecast(y, steps, SARIMA_CONFIG)
    mean_values = np.minimum(mean_values, CASH_NEED_CLIP_UPPER)

    if len(error_blocks) < MIN_BACKTEST_ERROR_BLOCKS:
        adjusted = np.minimum(
            mean_values * (1.0 + SARIMA_NO_ERROR_HISTORY_ADJUSTMENT),
            CASH_NEED_CLIP_UPPER,
        )
        return {
            QUANTILE_COLUMNS[float(quantile)]: adjusted.copy()
            for quantile in BOOTSTRAP_QUANTILES
        }

    sampled_indexes = random_generator.integers(
        low=0,
        high=len(error_blocks),
        size=BOOTSTRAP_ITERATIONS,
    )
    scenarios = np.minimum(
        mean_values.reshape(1, -1) + error_blocks[sampled_indexes],
        CASH_NEED_CLIP_UPPER,
    )
    quantile_values = np.quantile(scenarios, BOOTSTRAP_QUANTILES, axis=0)
    return {
        QUANTILE_COLUMNS[float(quantile)]: np.minimum(
            quantile_values[index], CASH_NEED_CLIP_UPPER
        )
        for index, quantile in enumerate(BOOTSTRAP_QUANTILES)
    }


def forecast_cashdesk(
    cashdesk_df: pd.DataFrame,
    report_date: pd.Timestamp,
    steps: int,
    random_generator: np.random.Generator,
) -> pd.DataFrame:
    saldo_series = make_regular_daily_series(
        cashdesk_df, "atdtmco_saldo_turn_fact", report_date
    )
    ns_series = make_regular_daily_series(
        cashdesk_df, "atdtmco_ns_fact", report_date
    )
    future_index = make_future_index(ns_series, steps)

    try:
        if saldo_series.notna().sum() < MIN_SARIMA_DAYS:
            raise ValueError("Короткая история saldo")
        _, saldo_values = guarded_sarima_forecast(
            saldo_series, steps, SARIMA_CONFIG
        )
    except Exception:
        saldo_values = weekday_point_forecast(saldo_series, steps)

    try:
        if ns_series.notna().sum() < MIN_SARIMA_DAYS:
            raise ValueError("Короткая история NS")
        ns_grid = forecast_ns_quantile_grid(
            ns_series, steps, random_generator
        )
    except Exception:
        ns_grid = {
            QUANTILE_COLUMNS[float(quantile)]: weekday_quantile_values(
                ns_series, steps, float(quantile)
            )
            for quantile in BOOTSTRAP_QUANTILES
        }

    result = pd.DataFrame(
        {
            "forecast_date": future_index,
            "atdtmco_saldo_turn_pred": saldo_values,
        }
    )
    for column_name, values in ns_grid.items():
        result[column_name] = values
    return result

## 4. Параллельный ретро-прогон и checkpoints

Один пул `loky` переиспользуется для всех дат; внутри каждой даты параллелятся кассы. Все загрузки Oracle, кэшей и checkpoints остаются в parent-процессе. RNG каждой задачи определяется только `RANDOM_SEED`, ordinal даты и стабильным индексом кассы в отсортированном списке.

Checkpoint записывается атомарно только после успешного завершения всей даты и содержит все 30 горизонтов и 15 квантильных колонок.

In [ ]:
def forecast_cashdesk_task(
    score_date: pd.Timestamp,
    report_date: pd.Timestamp,
    cashdesk_index: int,
    cashdesk_name: str,
    cashdesk_df: pd.DataFrame,
) -> pd.DataFrame:
    random_generator = np.random.default_rng(
        np.random.SeedSequence(
            [RANDOM_SEED, int(score_date.toordinal()), int(cashdesk_index)]
        )
    )
    translated_names = cashdesk_df["atdtmco_cashdesk_name_trn"].dropna()
    translated_name = translated_names.iloc[-1] if len(translated_names) else pd.NA
    forecast_df = forecast_cashdesk(
        cashdesk_df.sort_values("calday"),
        report_date,
        MODEL_STEPS,
        random_generator,
    )
    forecast_df = forecast_df[
        (forecast_df["forecast_date"] >= score_date)
        & (
            forecast_df["forecast_date"]
            < score_date + pd.Timedelta(days=FORECAST_DAYS)
        )
    ].copy()
    forecast_df.insert(0, "score_date", score_date)
    forecast_df.insert(1, "report_date", report_date)
    forecast_df.insert(2, "atdtmco_cashdesk_name", cashdesk_name)
    forecast_df.insert(3, "atdtmco_cashdesk_name_trn", translated_name)
    return forecast_df


def build_scoring_tasks(
    score_date: pd.Timestamp,
    all_daily_df: pd.DataFrame,
) -> Tuple[pd.Timestamp, List[Tuple[int, str, pd.DataFrame]]]:
    report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    history_date_from = report_date - pd.DateOffset(months=HISTORY_MONTHS)
    active_date_from = report_date - pd.DateOffset(months=ACTIVE_LOOKBACK_MONTHS)
    available_df = all_daily_df[
        (all_daily_df["calday"] >= history_date_from)
        & (all_daily_df["calday"] <= report_date)
    ]
    active_cashdesks = sorted(
        available_df.loc[
            available_df["calday"] >= active_date_from,
            "atdtmco_cashdesk_name",
        ].dropna().unique().tolist()
    )
    tasks = []
    for cashdesk_index, cashdesk_name in enumerate(active_cashdesks):
        cashdesk_df = available_df[
            available_df["atdtmco_cashdesk_name"] == cashdesk_name
        ].copy()
        tasks.append((cashdesk_index, cashdesk_name, cashdesk_df))
    return report_date, tasks


def validate_day_result(
    score_date: pd.Timestamp,
    day_result_df: pd.DataFrame,
) -> None:
    score_date = pd.Timestamp(score_date).normalize()
    missing_quantile_columns = set(QUANTILE_COLUMNS.values()) - set(
        day_result_df.columns
    )
    if missing_quantile_columns:
        raise ValueError("Checkpoint не содержит все квантильные колонки")
    if day_result_df.empty:
        raise ValueError("Пустой результат даты {}".format(score_date.date()))
    if not pd.to_datetime(day_result_df["score_date"]).eq(score_date).all():
        raise ValueError("В результате смешаны даты скоринга")

    expected_report_date = score_date - pd.Timedelta(days=DATA_LAG_DAYS)
    if not pd.to_datetime(day_result_df["report_date"]).eq(expected_report_date).all():
        raise ValueError("Checkpoint рассчитан с другим лагом данных")
    expected_last_forecast_date = score_date + pd.Timedelta(days=FORECAST_DAYS - 1)
    forecast_dates = pd.to_datetime(day_result_df["forecast_date"])
    if forecast_dates.min() != score_date or forecast_dates.max() != expected_last_forecast_date:
        raise ValueError("Checkpoint содержит неверный диапазон горизонта")

    key_columns = ["score_date", "atdtmco_cashdesk_name", "forecast_date"]
    if day_result_df.duplicated(key_columns).any():
        raise ValueError("Перед записью checkpoint найдены дубли")
    horizons_per_cashdesk = day_result_df.groupby(
        "atdtmco_cashdesk_name"
    )["forecast_date"].nunique()
    if not horizons_per_cashdesk.eq(FORECAST_DAYS).all():
        raise ValueError("Дата рассчитана не на все горизонты")

    prediction_columns = ["atdtmco_saldo_turn_pred"] + list(
        QUANTILE_COLUMNS.values()
    )
    if not np.isfinite(
        day_result_df[prediction_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Checkpoint содержит невалидные прогнозы")


def run_retro_scoring_day(
    score_date: pd.Timestamp,
    all_daily_df: pd.DataFrame,
    parallel: Parallel,
) -> pd.DataFrame:
    score_date = pd.Timestamp(score_date).normalize()
    report_date, tasks = build_scoring_tasks(score_date, all_daily_df)
    result_parts = parallel(
        delayed(forecast_cashdesk_task)(
            score_date,
            report_date,
            cashdesk_index,
            cashdesk_name,
            cashdesk_df,
        )
        for cashdesk_index, cashdesk_name, cashdesk_df in tasks
    )
    forecast_result_df = pd.concat(result_parts, ignore_index=True)

    actual_df = all_daily_df[
        (all_daily_df["calday"] >= score_date)
        & (
            all_daily_df["calday"]
            < score_date + pd.Timedelta(days=FORECAST_DAYS)
        )
    ][
        [
            "atdtmco_cashdesk_name",
            "atdtmco_cashdesk_name_trn",
            "calday",
            "atdtmco_saldo_turn_fact",
            "atdtmco_ns_daily_min_raw",
            "atdtmco_ns_fact",
        ]
    ].rename(columns={"calday": "forecast_date"})
    forecast_result_df = forecast_result_df.merge(
        actual_df,
        on=["atdtmco_cashdesk_name", "forecast_date"],
        how="left",
        suffixes=("", "_fact"),
    )
    forecast_result_df["atdtmco_cashdesk_name_trn"] = forecast_result_df[
        "atdtmco_cashdesk_name_trn_fact"
    ].combine_first(forecast_result_df["atdtmco_cashdesk_name_trn"])
    forecast_result_df = forecast_result_df.drop(
        columns="atdtmco_cashdesk_name_trn_fact"
    )
    fact_columns = [
        "atdtmco_saldo_turn_fact",
        "atdtmco_ns_daily_min_raw",
        "atdtmco_ns_fact",
    ]
    forecast_result_df[fact_columns] = forecast_result_df[fact_columns].fillna(0.0)
    validate_day_result(score_date, forecast_result_df)
    return forecast_result_df


CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
retro_result_parts = []
with parallel_backend("loky", inner_max_num_threads=1):
    with Parallel(n_jobs=N_JOBS) as parallel:
        for score_index, score_date in enumerate(score_dates, start=1):
            checkpoint_path = CHECKPOINT_DIR / "retro_{:%Y-%m-%d}.parquet".format(
                score_date
            )
            checkpoint_is_valid = False
            if USE_CHECKPOINTS and checkpoint_path.exists():
                try:
                    day_result_df = pd.read_parquet(checkpoint_path)
                    validate_day_result(score_date, day_result_df)
                    checkpoint_is_valid = True
                    status = "загружено"
                except Exception as checkpoint_error:
                    print(
                        "Checkpoint {} пересчитывается: {}".format(
                            score_date.date(), checkpoint_error
                        )
                    )
            if not checkpoint_is_valid:
                day_result_df = run_retro_scoring_day(
                    score_date, daily_df, parallel
                )
                temporary_path = checkpoint_path.with_suffix(".tmp.parquet")
                day_result_df.to_parquet(temporary_path, index=False)
                temporary_path.replace(checkpoint_path)
                status = "рассчитано"
            retro_result_parts.append(day_result_df)
            print("[{}/{}] {}: {}".format(
                score_index, len(score_dates), status, score_date.date()
            ))

retro_result_df = pd.concat(retro_result_parts, ignore_index=True)
print("Итог: {:,} строк, {:,} касс".format(
    len(retro_result_df),
    retro_result_df["atdtmco_cashdesk_name"].nunique(),
))

## 5. Ретро-калибровка квантиля

Сравнение выполняется по первому дню горизонта для строк, где есть старый прогноз. Доля невыдач старого прогноза считается на той же выборке. Автовыбор минимизирует абсолютное отклонение новой доли невыдач от старой; при равенстве выбирается меньший `MAE95`.

In [ ]:
def ns_mae95(
    evaluation_df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    clean_df = evaluation_df.dropna(subset=[fact_col, pred_col])
    absolute_error = (clean_df[fact_col] - clean_df[pred_col]).abs()
    trim_count = int(np.floor(len(clean_df) * 0.05))
    excluded_indexes = clean_df[fact_col].nsmallest(trim_count).index
    return float(absolute_error.drop(index=excluded_indexes).mean())


first_day_grid_df = retro_result_df[
    retro_result_df["forecast_date"] == retro_result_df["score_date"]
].copy()
matched_grid_df = first_day_grid_df.merge(
    old_history_dedup_df[
        ["cashdesk_name", "calday", "atdtmco_ns_pred_old"]
    ],
    left_on=["atdtmco_cashdesk_name_trn", "forecast_date"],
    right_on=["cashdesk_name", "calday"],
    how="inner",
).dropna(subset=["atdtmco_ns_fact", "atdtmco_ns_pred_old"])
if matched_grid_df.empty:
    raise RuntimeError("Нет сопоставленных строк со старым прогнозом")

old_breach_rate = (
    matched_grid_df["atdtmco_ns_fact"]
    < matched_grid_df["atdtmco_ns_pred_old"]
).mean()
quantile_rows = []
for quantile in BOOTSTRAP_QUANTILES:
    quantile_value = float(quantile)
    pred_col = QUANTILE_COLUMNS[quantile_value]
    clean_df = matched_grid_df.dropna(
        subset=["atdtmco_ns_fact", pred_col]
    ).copy()
    error = clean_df["atdtmco_ns_fact"] - clean_df[pred_col]
    absolute_error = error.abs()
    breach_rate = (clean_df["atdtmco_ns_fact"] < clean_df[pred_col]).mean()
    quantile_rows.append(
        {
            "квантиль": quantile_value,
            "количество строк": len(clean_df),
            "% невыдач": breach_rate,
            "отклонение от старого % невыдач": breach_rate - old_breach_rate,
            "MAE": absolute_error.mean(),
            "MAE95": ns_mae95(
                clean_df, "atdtmco_ns_fact", pred_col
            ),
            "медианная абсолютная ошибка": absolute_error.median(),
            "P90 абсолютной ошибки": absolute_error.quantile(0.90),
            "смещение факт − прогноз": error.mean(),
        }
    )
quantile_selection_df = pd.DataFrame(quantile_rows)

auto_quantile = float(
    quantile_selection_df.assign(
        _absolute_breach_deviation=quantile_selection_df[
            "отклонение от старого % невыдач"
        ].abs()
    )
    .sort_values(
        ["_absolute_breach_deviation", "MAE95", "квантиль"],
        kind="mergesort",
    )
    .iloc[0]["квантиль"]
)
if SELECTED_QUANTILE is None:
    selected_quantile = auto_quantile
else:
    matching_quantiles = [
        float(quantile)
        for quantile in BOOTSTRAP_QUANTILES
        if np.isclose(float(quantile), float(SELECTED_QUANTILE))
    ]
    if not matching_quantiles:
        raise ValueError("SELECTED_QUANTILE должен входить в BOOTSTRAP_QUANTILES")
    selected_quantile = matching_quantiles[0]
selected_quantile_column = QUANTILE_COLUMNS[selected_quantile]
retro_result_df["atdtmco_ns_pred"] = retro_result_df[
    selected_quantile_column
]

first_day_df = retro_result_df[
    retro_result_df["forecast_date"] == retro_result_df["score_date"]
].copy()
old_new_compare_df = first_day_df.merge(
    old_history_dedup_df[
        ["cashdesk_name", "calday", "atdtmco_ns_pred_old"]
    ],
    left_on=["atdtmco_cashdesk_name_trn", "forecast_date"],
    right_on=["cashdesk_name", "calday"],
    how="inner",
).dropna(subset=[
    "atdtmco_ns_fact", "atdtmco_ns_pred", "atdtmco_ns_pred_old"
])

quantile_formats = {
    "квантиль": "{:.2f}",
    "% невыдач": "{:.2%}",
    "отклонение от старого % невыдач": "{:+.2%}",
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
    "медианная абсолютная ошибка": "{:,.0f}",
    "P90 абсолютной ошибки": "{:,.0f}",
    "смещение факт − прогноз": "{:,.0f}",
}
print("Старый % невыдач на matched-выборке: {:.2%}".format(old_breach_rate))
print("Авто q={:.2f}; выбран q={:.2f} ({})".format(
    auto_quantile, selected_quantile, selected_quantile_column
))
display(quantile_selection_df.style.format(quantile_formats))

## 6. Итоговые метрики и четыре CSV

Метрики считаются только по всем дням, без смешения сегментов. Для NS сравниваются выбранный новый прогноз и старый `forecast_model` на общей matched-выборке; saldo оценивается для нового центрального прогноза. `MAE95` NS удаляет `floor(5%)` наиболее отрицательных фактов, а saldo — `floor(5%)` наибольших абсолютных фактов.

Сохраняются только: все горизонты выбранного прогноза, first-day сравнение, сводка метрик и таблица выбора квантиля.

In [ ]:
def median_smoothness_ratio(
    evaluation_df: pd.DataFrame,
    fact_col: str,
    pred_col: str,
) -> float:
    ratios = []
    for _, group_df in evaluation_df.groupby("atdtmco_cashdesk_name"):
        ordered_df = group_df.sort_values("forecast_date")
        consecutive_mask = ordered_df["forecast_date"].diff().dt.days.eq(1)
        fact_variation = ordered_df[fact_col].diff().abs()[consecutive_mask].sum()
        pred_variation = ordered_df[pred_col].diff().abs()[consecutive_mask].sum()
        if fact_variation > 0:
            ratios.append(pred_variation / fact_variation)
    return float(np.median(ratios)) if ratios else np.nan


def build_metric_row(
    evaluation_df: pd.DataFrame,
    target: str,
    model_name: str,
    calculation_level: str,
    cashdesk_name: str,
    fact_col: str,
    pred_col: str,
) -> Dict[str, object]:
    clean_df = evaluation_df.dropna(subset=[fact_col, pred_col]).copy()
    error = clean_df[fact_col] - clean_df[pred_col]
    absolute_error = error.abs()
    trim_count = int(np.floor(len(clean_df) * 0.05))
    if target == "NS":
        excluded_indexes = clean_df[fact_col].nsmallest(trim_count).index
        breach_rate = (clean_df[fact_col] < clean_df[pred_col]).mean()
        wape = np.nan
    else:
        excluded_indexes = clean_df[fact_col].abs().nlargest(trim_count).index
        breach_rate = np.nan
        denominator = clean_df[fact_col].abs().sum()
        wape = absolute_error.sum() / denominator if denominator > 0 else np.nan
    trimmed_absolute_error = absolute_error.drop(index=excluded_indexes)
    return {
        "показатель": target,
        "модель": model_name,
        "уровень расчёта": calculation_level,
        "касса": cashdesk_name,
        "количество строк": len(clean_df),
        "количество касс": clean_df["atdtmco_cashdesk_name"].nunique(),
        "количество дат скоринга": clean_df["score_date"].nunique(),
        "средний факт": clean_df[fact_col].mean(),
        "средний прогноз": clean_df[pred_col].mean(),
        "MAE": absolute_error.mean(),
        "MAE95": trimmed_absolute_error.mean(),
        "медианная абсолютная ошибка": absolute_error.median(),
        "P90 абсолютной ошибки": absolute_error.quantile(0.90),
        "смещение факт − прогноз": error.mean(),
        "% невыдач": breach_rate,
        "% абсолютной ошибки (WAPE)": wape,
        "медианный коэффициент сглаженности": median_smoothness_ratio(
            clean_df, fact_col, pred_col
        ),
    }


def append_level_metrics(
    rows: List[Dict[str, object]],
    evaluation_df: pd.DataFrame,
    target: str,
    models: List[Tuple[str, str]],
    fact_col: str,
) -> None:
    for model_name, pred_col in models:
        rows.append(
            build_metric_row(
                evaluation_df,
                target,
                model_name,
                "в целом",
                "ВСЕ",
                fact_col,
                pred_col,
            )
        )
    for cashdesk_name, cashdesk_df in evaluation_df.groupby(
        "atdtmco_cashdesk_name"
    ):
        for model_name, pred_col in models:
            rows.append(
                build_metric_row(
                    cashdesk_df,
                    target,
                    model_name,
                    "по кассе",
                    cashdesk_name,
                    fact_col,
                    pred_col,
                )
            )


metric_rows = []
append_level_metrics(
    metric_rows,
    old_new_compare_df,
    "NS",
    [
        ("новый выбранный квантиль", "atdtmco_ns_pred"),
        ("старый forecast_model", "atdtmco_ns_pred_old"),
    ],
    "atdtmco_ns_fact",
)
append_level_metrics(
    metric_rows,
    first_day_df,
    "saldo_turn",
    [("новый guarded SARIMA", "atdtmco_saldo_turn_pred")],
    "atdtmco_saldo_turn_fact",
)
metrics_summary_df = pd.DataFrame(metric_rows)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
selected_forecast_columns = [
    "score_date",
    "report_date",
    "atdtmco_cashdesk_name",
    "atdtmco_cashdesk_name_trn",
    "forecast_date",
    "atdtmco_saldo_turn_pred",
    "atdtmco_ns_pred",
    "atdtmco_saldo_turn_fact",
    "atdtmco_ns_fact",
]
selected_forecasts_df = retro_result_df[selected_forecast_columns].copy()
first_day_comparison_columns = selected_forecast_columns + [
    "cashdesk_name",
    "atdtmco_ns_pred_old",
]
first_day_comparison_df = old_new_compare_df[
    first_day_comparison_columns
].copy()

selected_forecasts_df.to_csv(
    OUTPUT_DIR / "all_horizons_selected_forecasts.csv", index=False
)
first_day_comparison_df.to_csv(
    OUTPUT_DIR / "first_day_old_new_comparison.csv", index=False
)
metrics_summary_df.to_csv(OUTPUT_DIR / "metrics_summary.csv", index=False)
quantile_selection_df.to_csv(
    OUTPUT_DIR / "quantile_selection.csv", index=False
)

metric_formats = {
    "средний факт": "{:,.0f}",
    "средний прогноз": "{:,.0f}",
    "MAE": "{:,.0f}",
    "MAE95": "{:,.0f}",
    "медианная абсолютная ошибка": "{:,.0f}",
    "P90 абсолютной ошибки": "{:,.0f}",
    "смещение факт − прогноз": "{:,.0f}",
    "% невыдач": "{:.2%}",
    "% абсолютной ошибки (WAPE)": "{:.2%}",
    "медианный коэффициент сглаженности": "{:.2f}",
}
display(
    metrics_summary_df[
        metrics_summary_df["уровень расчёта"] == "в целом"
    ].style.format(metric_formats, na_rep="—")
)
display(
    metrics_summary_df[
        metrics_summary_df["уровень расчёта"] == "по кассе"
    ].style.format(metric_formats, na_rep="—")
)

## 7. Слайд: алгоритм и ручная выгрузка

```text
12 месяцев истории до T−2
           │
  дневные ряды каждой кассы
  (исключённый период = NaN)
           │
   nan-guarded SARIMA(1,1,1)×(1,0,1,7)
           │
     ┌─────┴─────┐
     │           │
 saldo_turn      NS
 central         rolling guarded backtest
 forecast        + 1000 сценариев один раз
                 + q01..q15
                       │
                 ретро-калибровка q:
                 ближе всего к old breach,
                 tie-break по MAE95
                       │
                 выбранный NS-прогноз
```

Квантиль не зафиксирован на 15%: он выбирается ретро-калибровкой под фактическую долю невыдач старого алгоритма на той же matched-выборке.

### Ручная запись в Oracle

Следующая ячейка выключена по умолчанию. При `WRITE_TO_ORACLE=True` она добавляет в `EMA_CASHDESK_PREDS_2M` **все даты скоринга × 30 дней каждого горизонта × все рассчитанные кассы** для выбранного квантиля. Перед `append` проверяются дубликаты и конечность числовых значений.

In [ ]:
# РУЧНОЙ ШАГ: переключить на True только после проверки результатов.
WRITE_TO_ORACLE = False

if WRITE_TO_ORACLE:
    oracle_export_df = retro_result_df[
        [
            "score_date",
            "report_date",
            "atdtmco_cashdesk_name",
            "forecast_date",
            "atdtmco_saldo_turn_pred",
            "atdtmco_ns_pred",
            "atdtmco_saldo_turn_fact",
            "atdtmco_ns_fact",
        ]
    ].copy()
    oracle_key_columns = [
        "score_date",
        "atdtmco_cashdesk_name",
        "forecast_date",
    ]
    oracle_numeric_columns = [
        "atdtmco_saldo_turn_pred",
        "atdtmco_ns_pred",
        "atdtmco_saldo_turn_fact",
        "atdtmco_ns_fact",
    ]
    if oracle_export_df.duplicated(oracle_key_columns).any():
        raise ValueError("Перед записью в Oracle найдены дубли")
    if not np.isfinite(
        oracle_export_df[oracle_numeric_columns].to_numpy(dtype=float)
    ).all():
        raise ValueError("Перед записью в Oracle найдены невалидные числа")

    oracle_export_df[oracle_numeric_columns] = oracle_export_df[
        oracle_numeric_columns
    ].round(2)
    if engine_cdw is None:
        engine_cdw = await create_cdw_engine()
    oracle.write(
        oracle_export_df,
        engine_cdw,
        ORACLE_TARGET_TABLE,
        batch_size=100_000,
        if_exists="append",
    )
    print("В {} добавлено {:,} строк".format(
        ORACLE_TARGET_TABLE, len(oracle_export_df)
    ))
else:
    print("Запись в Oracle отключена: WRITE_TO_ORACLE = False")